# Matched B-like vs C1 vs C4 Ablation — Google Colab

This notebook isolates two questions with **one shared set of four generated candidate utterances per test example**:

1. **B-like:** candidate 1 only → emotion label (**no dialogue history in the label prompt**)
2. **C1:** full history + candidate 1 → emotion label
3. **C4:** full history + all four candidates → emotion label

Because C1 and C4 reuse the same generated candidates, the only difference between them is the number of candidates shown to the final classifier. This is the clean comparison for testing whether multiple simulated futures help.

The notebook uploads `IEMOCAP_features.pkl`, uses Qwen2.5-7B-Instruct in 4-bit on Colab, checkpoints every 10 examples, and automatically downloads a ZIP at the end.


## 1. Install dependencies

In [ ]:
!pip -q install "transformers==4.46.3" "accelerate==1.1.1" "bitsandbytes==0.44.1" "safetensors==0.4.5"


## 2. Upload IEMOCAP pickle and configure the run

In [ ]:
from google.colab import files
from pathlib import Path
from dataclasses import dataclass
from collections import Counter
import os, json, re, pickle, random, shutil, gc
import torch

print("Upload IEMOCAP_features.pkl")
uploaded = files.upload()
assert uploaded, "No file uploaded."
DATA_PATH = str(Path(next(iter(uploaded.keys()))).resolve())

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
OUTPUT_ROOT = Path("/content/matched_candidate_ablation")
MAX_SAMPLES = 0          # 0 = all 1,592 canonical test points
MAX_HISTORY_TURNS = 10
N_CANDIDATES = 4
CHECKPOINT_EVERY = 10
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), "Enable a GPU runtime: Runtime > Change runtime type > T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))
print("Data:", DATA_PATH)
print("Output:", OUTPUT_ROOT)


## 3. Canonical IEMOCAP loader and evaluation

In [ ]:
EMOTION_LABELS = ["neutral", "frustration", "sadness", "anger", "excited", "happiness"]
LABEL2ID = {x: i for i, x in enumerate(EMOTION_LABELS)}
NUMERIC_TO_LABEL = {0: "happiness", 1: "sadness", 2: "neutral", 3: "anger", 4: "excited", 5: "frustration"}
NORMALIZE = {
    "neutral": "neutral", "neu": "neutral",
    "frustration": "frustration", "frustrated": "frustration", "fru": "frustration",
    "sadness": "sadness", "sad": "sadness",
    "anger": "anger", "angry": "anger", "ang": "anger",
    "excited": "excited", "excitement": "excited", "exc": "excited",
    "happiness": "happiness", "happy": "happiness", "hap": "happiness",
}

@dataclass
class Sample:
    dialogue_id: str
    history: list
    history_speakers: list
    history_emotions: list
    target_speaker: str
    target_emotion: str
    target_emotion_id: int

def normalize_label(x):
    if isinstance(x, int):
        return NUMERIC_TO_LABEL.get(x)
    try:
        if hasattr(x, "item"):
            value = x.item()
            if isinstance(value, int):
                return NUMERIC_TO_LABEL.get(value)
    except Exception:
        pass
    return NORMALIZE.get(str(x).strip().lower())

def speaker_label(x):
    if isinstance(x, (list, tuple)):
        return "A" if max(range(len(x)), key=lambda i: x[i]) == 0 else "B"
    try:
        # Supports NumPy arrays without importing NumPy directly.
        if hasattr(x, "shape") and len(x.shape) > 0:
            values = x.tolist()
            return "A" if max(range(len(values)), key=lambda i: values[i]) == 0 else "B"
    except Exception:
        pass
    return str(x)

def carve_val(train_vids, n_val=20):
    ordered = sorted(train_vids)
    val = set(ordered[-n_val:])
    return set(ordered[:-n_val]), val

def emit_samples(vid, utterances, speakers, labels):
    speakers = [speaker_label(s) for s in speakers]
    labels = [normalize_label(x) for x in labels]
    rows = []
    for t in range(1, len(utterances)):
        if t >= len(labels) or labels[t] is None:
            continue
        rows.append(Sample(
            dialogue_id=f"{vid}_t{t}",
            history=list(utterances[:t]),
            history_speakers=list(speakers[:t]),
            history_emotions=list(labels[:t]),
            target_speaker=speakers[t],
            target_emotion=labels[t],
            target_emotion_id=LABEL2ID[labels[t]],
        ))
    return rows

def load_canonical_iemocap(path):
    with open(path, "rb") as f:
        raw = pickle.load(f, encoding="latin1")

    if isinstance(raw, (list, tuple)) and len(raw) >= 9:
        video_speakers, video_labels, video_sentence = raw[1], raw[2], raw[6]
        train_vids, test_vids = list(raw[7]), set(raw[8])
    elif isinstance(raw, dict) and ("videoSentence" in raw or "trainVid" in raw):
        video_speakers = raw["videoSpeakers"]
        video_labels = raw["videoLabels"]
        video_sentence = raw["videoSentence"]
        train_vids, test_vids = list(raw["trainVid"]), set(raw["testVid"])
    else:
        raise ValueError("Expected the standard DialogueRNN-style IEMOCAP pickle.")

    train_set, dev_set = carve_val(train_vids, 20)
    splits = {"train": [], "dev": [], "test": []}
    for vid, utts in video_sentence.items():
        if vid in test_vids:
            split = "test"
        elif vid in dev_set:
            split = "dev"
        elif vid in train_set:
            split = "train"
        else:
            continue
        splits[split].extend(emit_samples(vid, utts, video_speakers[vid], video_labels[vid]))
    return splits

def format_history(s, max_turns=MAX_HISTORY_TURNS):
    rows = []
    for spk, emo, utt in zip(
        s.history_speakers[-max_turns:],
        s.history_emotions[-max_turns:],
        s.history[-max_turns:],
    ):
        rows.append(f"Speaker {spk} ({emo}): {utt}")
    return "\n".join(rows)

def parse_emotion(text):
    low = str(text).lower().strip()
    aliases = {
        "neutral": "neutral",
        "frustration": "frustration",
        "frustrated": "frustration",
        "sadness": "sadness",
        "sad": "sadness",
        "anger": "anger",
        "angry": "anger",
        "excited": "excited",
        "excitement": "excited",
        "happiness": "happiness",
        "happy": "happiness",
    }
    hits = []
    for alias, normalized in aliases.items():
        for match in re.finditer(re.escape(alias), low):
            hits.append((match.start(), normalized))
    return max(hits, key=lambda x: x[0])[1] if hits else None

def same_speaker_shift(s):
    for spk, emo in zip(reversed(s.history_speakers), reversed(s.history_emotions)):
        if spk == s.target_speaker and emo is not None:
            return s.target_emotion != emo
    return None

def f1_for_label(y_true, y_pred, label):
    tp = sum(t == label and p == label for t, p in zip(y_true, y_pred))
    fp = sum(t != label and p == label for t, p in zip(y_true, y_pred))
    fn = sum(t == label and p != label for t, p in zip(y_true, y_pred))
    denom = 2 * tp + fp + fn
    return 0.0 if denom == 0 else (2 * tp) / denom

def metric_bundle(y_true, y_pred):
    n = len(y_true)
    supports = Counter(y_true)
    per_class = {label: f1_for_label(y_true, y_pred, label) for label in EMOTION_LABELS}
    macro = sum(per_class.values()) / len(EMOTION_LABELS)
    weighted = sum(per_class[label] * supports[label] for label in EMOTION_LABELS) / max(n, 1)
    accuracy = sum(t == p for t, p in zip(y_true, y_pred)) / max(n, 1)
    return {"weighted_f1": weighted, "macro_f1": macro, "accuracy": accuracy, "per_class_f1": per_class}

def evaluate_records(records, samples, title, save_path):
    by_id = {r["dialogue_id"]: r for r in records}
    missing = [s.dialogue_id for s in samples if s.dialogue_id not in by_id]
    assert not missing, f"Missing predictions, first few: {missing[:5]}"

    ordered = [by_id[s.dialogue_id] for s in samples]
    y_true = [s.target_emotion for s in samples]
    y_pred = [r.get("predicted_emotion") for r in ordered]
    overall = metric_bundle(y_true, y_pred)

    es_idx = [i for i, s in enumerate(samples) if same_speaker_shift(s) is True]
    ns_idx = [i for i, s in enumerate(samples) if same_speaker_shift(s) is False]

    def subset(idx):
        return metric_bundle([y_true[i] for i in idx], [y_pred[i] for i in idx])

    metrics = {
        "title": title,
        "n": len(samples),
        **overall,
        "parse_failures": sum(p not in EMOTION_LABELS for p in y_pred),
        "es_n": len(es_idx),
        "es_metrics": subset(es_idx),
        "no_shift_n": len(ns_idx),
        "no_shift_metrics": subset(ns_idx),
        "undefined_n": len(samples) - len(es_idx) - len(ns_idx),
    }
    payload = {"metrics": metrics, "predictions": ordered}
    Path(save_path).write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    print(json.dumps(metrics, indent=2))
    print("Saved:", save_path)
    return metrics

splits = load_canonical_iemocap(DATA_PATH)
all_test_samples = splits["test"]
assert len(all_test_samples) == 1592, f"Expected 1592 canonical test points, found {len(all_test_samples)}"
samples = all_test_samples if MAX_SAMPLES == 0 else all_test_samples[:MAX_SAMPLES]
print({k: len(v) for k, v in splits.items()})
print("Run size:", len(samples))

## 4. Load Qwen2.5-7B-Instruct in 4-bit

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

gc.collect()
torch.cuda.empty_cache()

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)
model.eval()

print("Loaded:", MODEL_ID)
print("Allocated GB:", round(torch.cuda.memory_allocated() / 1024**3, 2))


## 5. Shared generation and strictly matched prompts

In [ ]:
from tqdm.auto import tqdm

def generate_texts(prompts, max_new_tokens):
    outputs = []
    for prompt in prompts:
        rendered = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True,
        )
        inputs = tokenizer(
            rendered, return_tensors="pt", truncation=True, max_length=3072
        ).to(model.device)
        with torch.inference_mode():
            generated = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        new_tokens = generated[:, inputs.input_ids.shape[1]:]
        outputs.append(tokenizer.decode(new_tokens[0], skip_special_tokens=True))
        del inputs, generated, new_tokens
    return outputs

def load_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def append_jsonl(path, rows):
    with Path(path).open("a", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def prompt_generate_four(s):
    return f"""Generate exactly four distinct plausible next utterances for Speaker {s.target_speaker}.
These are hypothetical possibilities, not claims about the real unseen future.

OUTPUT RULES:
- Output exactly four numbered, one-line utterances.
- Write only words the speaker could actually say aloud.
- Do not include emotion names, emotion labels, speaker names, rationales, stage directions, parenthetical descriptions, or metadata.
- Do not reveal or guess the gold emotion.

Dialogue history:
{format_history(s)}

Format:
1. utterance text only
2. utterance text only
3. utterance text only
4. utterance text only"""

def strip_metadata(text):
    x = str(text).strip()
    x = re.sub(r"^(?:candidate|utterance)\s*\d*\s*:\s*", "", x, flags=re.I)
    x = re.sub(r"^(?:speaker\s*)?[A-Za-z0-9_-]+\s*(?:\([^)]*\)|\[[^]]*\])?\s*:\s*", "", x, flags=re.I)
    x = re.sub(r"^(?:\([^)]*\)|\[[^]]*\])\s*:?\s*", "", x)
    return x.strip()

def parse_four(text):
    rows = []
    for line in str(text).splitlines():
        m = re.match(r"\s*\d+[.)]\s*(.+)", line)
        if m:
            c = strip_metadata(m.group(1))
            if c and c not in rows:
                rows.append(c)
    return rows[:4]

def label_instruction():
    return f"Valid emotions: {', '.join(EMOTION_LABELS)}\nReply exactly as <emotion>one_valid_label</emotion>."

def prompt_b_like(candidate):
    return f"""Classify the emotion expressed by this hypothetical next utterance.
Do not use any dialogue history.
{label_instruction()}

Utterance:
{candidate}"""

def prompt_c1(s, candidate):
    return f"""Forecast Speaker {s.target_speaker}'s next emotion.
The dialogue history is primary evidence. The one candidate is an uncertain simulated possibility, not the true unseen utterance.
{label_instruction()}

Dialogue history:
{format_history(s)}

Hypothetical candidate future:
1. {candidate}"""

def prompt_c4(s, candidates):
    block = "\n".join(f"{i+1}. {c}" for i, c in enumerate(candidates))
    return f"""Forecast Speaker {s.target_speaker}'s next emotion.
The dialogue history is primary evidence. The four candidates are uncertain simulated possibilities, not the true unseen utterance.
Consider all four possibilities; do not assume any candidate is the real future.
{label_instruction()}

Dialogue history:
{format_history(s)}

Hypothetical candidate futures:
{block}"""


## 6. Smoke test

In [ ]:
smoke = samples[:2]
raw_candidates = generate_texts([prompt_generate_four(s) for s in smoke], 160)
candidate_lists = [parse_four(x) for x in raw_candidates]
assert all(len(x) == 4 for x in candidate_lists), candidate_lists

for s, candidates in zip(smoke, candidate_lists):
    prompts = [prompt_b_like(candidates[0]), prompt_c1(s, candidates[0]), prompt_c4(s, candidates)]
    raw = generate_texts(prompts, 16)
    parsed = [parse_emotion(x) for x in raw]
    assert all(x in EMOTION_LABELS for x in parsed), list(zip(raw, parsed))
    print(s.dialogue_id, candidates, parsed)
print("SMOKE TEST PASSED")


## 7. Full resumable matched run

In [ ]:
JSONL = OUTPUT_ROOT / "matched_predictions.jsonl"
DONE = {r["dialogue_id"] for r in load_jsonl(JSONL)}
pending = [s for s in samples if s.dialogue_id not in DONE]
print("Already complete:", len(DONE), "Pending:", len(pending))

for start in tqdm(range(0, len(pending), CHECKPOINT_EVERY)):
    chunk = pending[start:start + CHECKPOINT_EVERY]

    raw_candidate_outputs = generate_texts([prompt_generate_four(s) for s in chunk], 160)
    candidate_lists = [parse_four(x) for x in raw_candidate_outputs]
    bad = [(s.dialogue_id, raw, c) for s, raw, c in zip(chunk, raw_candidate_outputs, candidate_lists) if len(c) != 4]
    if bad:
        raise RuntimeError(f"Candidate parse failure: {bad[0]}")

    rows = []
    for s, raw_candidates, candidates in zip(chunk, raw_candidate_outputs, candidate_lists):
        prompts = [
            prompt_b_like(candidates[0]),
            prompt_c1(s, candidates[0]),
            prompt_c4(s, candidates),
        ]
        raw_labels = generate_texts(prompts, 16)
        labels = [parse_emotion(x) for x in raw_labels]
        rows.append({
            "dialogue_id": s.dialogue_id,
            "gold_emotion": s.target_emotion,
            "is_emotion_shift": same_speaker_shift(s),
            "candidate_raw": raw_candidates,
            "candidates": candidates,
            "candidate_1": candidates[0],
            "b_like_predicted_emotion": labels[0],
            "b_like_raw": raw_labels[0],
            "c1_predicted_emotion": labels[1],
            "c1_raw": raw_labels[1],
            "c4_predicted_emotion": labels[2],
            "c4_raw": raw_labels[2],
        })
    append_jsonl(JSONL, rows)

records = load_jsonl(JSONL)
assert len(records) == len(samples), f"Incomplete: {len(records)}/{len(samples)}"
print("Generation complete:", len(records))


## 8. Evaluate all three arms and save comparison

In [ ]:
def evaluate_arm(records, samples, pred_key, title):
    by_id = {r["dialogue_id"]: r for r in records}
    ordered = [by_id[s.dialogue_id] for s in samples]
    y_true = [s.target_emotion for s in samples]
    y_pred = [r.get(pred_key) for r in ordered]
    overall = metric_bundle(y_true, y_pred)
    es_idx = [i for i, s in enumerate(samples) if same_speaker_shift(s) is True]
    ns_idx = [i for i, s in enumerate(samples) if same_speaker_shift(s) is False]
    subset = lambda idx: metric_bundle([y_true[i] for i in idx], [y_pred[i] for i in idx])
    return {
        "title": title, "n": len(samples), **overall,
        "parse_failures": sum(p not in EMOTION_LABELS for p in y_pred),
        "es_n": len(es_idx), "es_metrics": subset(es_idx),
        "no_shift_n": len(ns_idx), "no_shift_metrics": subset(ns_idx),
        "undefined_n": len(samples) - len(es_idx) - len(ns_idx),
    }

results = {
    "design": {
        "shared_candidate_generation": True,
        "candidate_1_reused_in_b_like_and_c1": True,
        "only_c1_vs_c4_difference": "one candidate versus four candidates in final prompt",
        "model": MODEL_ID,
        "decoding": "greedy_do_sample_false",
    },
    "b_like_candidate_only": evaluate_arm(records, samples, "b_like_predicted_emotion", "B-like: candidate 1 only -> emotion"),
    "c1_history_plus_one": evaluate_arm(records, samples, "c1_predicted_emotion", "C1: history + candidate 1 -> emotion"),
    "c4_history_plus_four": evaluate_arm(records, samples, "c4_predicted_emotion", "C4: history + four candidates -> emotion"),
}

RESULTS_PATH = OUTPUT_ROOT / "matched_results.json"
RESULTS_PATH.write_text(json.dumps({"results": results, "predictions": records}, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(results, indent=2))
print("Saved:", RESULTS_PATH)


## 9. ZIP and download directly to your computer

In [ ]:
README = OUTPUT_ROOT / "README.txt"
README.write_text(
    "Matched Qwen2.5-7B candidate ablation.\n"
    "B-like: candidate 1 only, no history in label prompt.\n"
    "C1: history + the same candidate 1.\n"
    "C4: history + all four candidates from the same generation.\n"
    "C1 vs C4 isolates candidate count. B-like vs C1 isolates access to history.\n",
    encoding="utf-8",
)
ZIP_PATH = shutil.make_archive("/content/matched_candidate_ablation_results", "zip", root_dir=OUTPUT_ROOT)
print("ZIP:", ZIP_PATH)
files.download(ZIP_PATH)
